In [ ]:
# Save models locally
import os
os.makedirs('../models/quality_inspection', exist_ok=True)

best_model_obj = models_to_log[best_model_name][0]

if best_model_name == 'ANN':
    best_model_obj.save('../models/quality_inspection/best_ann_model.h5')
    print(f"Saved ANN model to ../models/quality_inspection/best_ann_model.h5")
else:
    joblib.dump(best_model_obj, '../models/quality_inspection/best_sklearn_model.pkl')
    print(f"Saved sklearn model to ../models/quality_inspection/best_sklearn_model.pkl")

# Save feature engineer and scaler
joblib.dump(fe_train, '../models/quality_inspection/feature_engineer.pkl')
joblib.dump(scaler, '../models/quality_inspection/scaler.pkl')
print("Saved preprocessing objects")

# Create summary report
summary = f"""
# Quality Inspection Model Training Summary

## Dataset
- Steel Plates Faults (7 defect classes)
- Total samples: {len(df_clean)}
- Training samples: {len(X_train)}
- Test samples: {len(X_test)}

## Best Model: {best_model_name}
- Accuracy: {comparison_df.iloc[0]['accuracy']:.4f}
- Precision (weighted): {comparison_df.iloc[0]['precision_weighted']:.4f}
- Recall (weighted): {comparison_df.iloc[0]['recall_weighted']:.4f}
- F1 Score (weighted): {comparison_df.iloc[0]['f1_weighted']:.4f}

## Model Comparison
{comparison_df.to_string()}

## Next Steps
1. Deploy best model to production
2. Set up monitoring for model drift
3. Implement A/B testing framework
4. Plan for model retraining schedule
"""

with open('../models/quality_inspection/SUMMARY.md', 'w') as f:
    f.write(summary)

print("\nSummary Report:\n")
print(summary)

## Step 13: Model Persistence & Summary

In [ ]:
# Plot confusion matrices for best models
best_model_name = comparison_df.iloc[0]['model']
print(f"Best Model: {best_model_name}\n")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

models_cm = [
    ('Decision Tree', y_pred_dt),
    ('SVM', y_pred_svm),
    ('XGBoost', y_pred_xgb),
    ('ANN', y_pred_ann_class)
]

for idx, (name, y_pred) in enumerate(models_cm):
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{name} - Confusion Matrix')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

# Best model detailed report
best_preds = models_to_log[best_model_name][1]
print(f"Detailed Report for {best_model_name}:")
print(classification_report(y_test, best_preds, target_names=[f'Class {i}' for i in range(1, 8)]))

## Step 12: Confusion Matrix & Classification Reports

In [ ]:
# Configure MLflow
mlflow.set_experiment("Quality-Inspection-Steel-Plates")

# Log each model to MLflow
models_to_log = {
    'Decision Tree': (dt_pipeline, y_pred_dt),
    'SVM': (svm_pipeline, y_pred_svm),
    'XGBoost': (xgb_pipeline, y_pred_xgb),
    'ANN': (ann_model, y_pred_ann_class)
}

for model_name, (model, predictions) in models_to_log.items():
    with mlflow.start_run(run_name=model_name):
        metrics = compute_metrics(y_test, predictions)
        
        # Log metrics
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)
        
        # Log params
        mlflow.log_param('dataset', 'Steel Plates Faults')
        mlflow.log_param('test_size', 0.2)
        mlflow.log_param('random_state', 42)
        
        # Log model
        if isinstance(model, Sequential):
            mlflow.keras.log_model(model, 'model')
        else:
            mlflow.sklearn.log_model(model, 'model')
        
        print(f"Logged {model_name} to MLflow")

print("\nExperiments logged to MLflow!")

## Step 11: MLflow Integration

In [ ]:
# Compute all metrics for comparison
def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

comparison_data = {
    'model': ['Decision Tree', 'SVM', 'XGBoost', 'ANN'],
    **{k: [] for k in ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']}
}

for name, y_pred in [
    ('Decision Tree', y_pred_dt),
    ('SVM', y_pred_svm),
    ('XGBoost', y_pred_xgb),
    ('ANN', y_pred_ann_class)
]:
    metrics = compute_metrics(y_test, y_pred)
    for k, v in metrics.items():
        comparison_data[k].append(v)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('f1_weighted', ascending=False).reset_index(drop=True)

print("Model Comparison:")
print(comparison_df.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
metrics = comparison_df.iloc[:, 1:].T
metrics.columns = comparison_df['model']
metrics.plot(kind='bar', ax=ax)
ax.set_title('Model Comparison - All Metrics')
ax.set_ylabel('Score')
ax.set_xlabel('Metric')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 10: Model Comparison

In [ ]:
# Create validation split from training data
n_train = len(X_train_scaled)
val_size = int(0.2 * n_train)
X_train_nn = X_train_scaled[:-val_size]
X_val_nn = X_train_scaled[-val_size:]
y_train_nn = y_train.values[:-val_size]
y_val_nn = y_train.values[-val_size:]

# Build ANN
ann_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_nn.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(7, activation='softmax')
])

ann_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train with early stopping
callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = ann_model.fit(
    X_train_nn, y_train_nn,
    validation_data=(X_val_nn, y_val_nn),
    epochs=50,
    batch_size=32,
    callbacks=[callback],
    verbose=0
)

y_pred_ann = ann_model.predict(X_test_scaled, verbose=0)
y_pred_ann_class = np.argmax(y_pred_ann, axis=1) + 1  # Classes are 1-7

ann_accuracy = accuracy_score(y_test.values, y_pred_ann_class)
ann_f1 = f1_score(y_test.values, y_pred_ann_class, average='weighted')

print(f"ANN Results:")
print(f"  Accuracy: {ann_accuracy:.4f}")
print(f"  F1 Score (weighted): {ann_f1:.4f}")
print(f"  Epochs Trained: {len(history.history['loss'])}")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('ANN Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('ANN Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 9: ANN Model

In [ ]:
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        num_class=7,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

xgb_pipeline.fit(X_train_eng, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test_eng)

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"XGBoost Results:")
print(f"  Accuracy: {xgb_accuracy:.4f}")
print(f"  F1 Score (weighted): {xgb_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

## Step 8: XGBoost Model

In [ ]:
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        probability=True,
        random_state=42
    ))
])

svm_pipeline.fit(X_train_eng, y_train)
y_pred_svm = svm_pipeline.predict(X_test_eng)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm, average='weighted')

print(f"SVM Results:")
print(f"  Accuracy: {svm_accuracy:.4f}")
print(f"  F1 Score (weighted): {svm_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

## Step 7: SVM Model

In [ ]:
dt_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    ))
])

dt_pipeline.fit(X_train_eng, y_train)
y_pred_dt = dt_pipeline.predict(X_test_eng)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt, average='weighted')

print(f"Decision Tree Results:")
print(f"  Accuracy: {dt_accuracy:.4f}")
print(f"  F1 Score (weighted): {dt_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

## Step 6: Decision Tree Model

In [ ]:
# Split before feature engineering to prevent data leakage
X = df_clean[feature_cols]
y = df_clean['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

# Fit feature engineer on training data only
fe_train = FeatureEngineer()
X_train_eng = fe_train.fit_transform(X_train)
X_test_eng = fe_train.transform(X_test)

print(f"\nEngineered feature shape: {X_train_eng.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_eng)
X_test_scaled = scaler.transform(X_test_eng)

print(f"Scaled feature shape: {X_train_scaled.shape}")

## Step 5: Train/Test Split

In [ ]:
class FeatureEngineer:
    """Implement fit-transform pattern for feature engineering to prevent data leakage."""
    
    def __init__(self):
        self.pixel_area_mean = 1.0
        self.pixel_area_std = 1.0
        self._is_fitted = False
    
    def fit(self, df: pd.DataFrame):
        """Fit engineer on training data."""
        self.pixel_area_mean = df['Pixel_area'].mean()
        self.pixel_area_std = max(df['Pixel_area'].std(), 1.0)
        self._is_fitted = True
        return self
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform using fitted statistics."""
        if not self._is_fitted:
            raise ValueError("Fit first!")
        
        df = df.copy()
        df['x_range'] = df['X_Maximum'] - df['X_Minimum']
        df['y_range'] = df['Y_Maximum'] - df['Y_Minimum']
        df['area_ratio'] = df['Pixel_area'] / max(self.pixel_area_mean, 1.0)
        df['nuclei_density'] = df['Bare_Nuclei'] / df['Pixel_area'].replace(0, 1.0)
        df['shape_ratio'] = df['x_range'] / df['y_range'].replace(0, 1.0)
        return df
    
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit and transform in one call."""
        return self.fit(df).transform(df)

# Example usage
fe = FeatureEngineer()
df_engineered = fe.fit_transform(df_clean[feature_cols])
print(f"Engineered features shape: {df_engineered.shape}")
print(f"\nFirst few rows:\n{df_engineered.head()}")
print(f"\nNew features statistics:\n{df_engineered[['x_range', 'y_range', 'area_ratio', 'nuclei_density', 'shape_ratio']].describe()}")

## Step 4: Feature Engineering

In [ ]:
# Correlation matrix
correlation = df_clean[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print(f"\nCorrelation Statistics:\n{correlation.describe()}")

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for idx, col in enumerate(feature_cols):
    ax = axes[idx // 3, idx % 3]
    df_clean[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_clean['class'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Defect Class Distribution')
axes[0].set_xlabel('Fault Class')
axes[0].set_ylabel('Count')

df_clean['class'].value_counts(normalize=True).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Defect Class Distribution (%)')
axes[1].set_xlabel('Fault Class')
axes[1].set_ylabel('Proportion')

plt.tight_layout()
plt.show()

# Class imbalance ratio
class_counts = df_clean['class'].value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nClass Imbalance Ratio: {imbalance_ratio:.2f}")
print(f"Class distribution:\n{class_counts}")

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean dataset: remove duplicates, handle missing values."""
    df = df.copy()
    df = df.drop_duplicates(keep='first')
    df = df.dropna()
    return df

df_clean = clean_data(df)
print(f"After cleaning - shape: {df_clean.shape}")
print(f"Duplicates removed: {len(df) - len(df_clean)}")
print(f"Missing values:\n{df_clean.isnull().sum()}")

## Step 2: Data Cleaning & Preprocessing

In [ ]:
# Load the Steel Plates Faults dataset
# Using sample synthetic data for demonstration
np.random.seed(42)

n_samples = 1000
feature_cols = ['X_Minimum', 'X_Maximum', 'Y_Minimum', 'Y_Maximum', 'Pixel_area', 'Bare_Nuclei']

# Generate synthetic Steel Plates data (7 defect classes)
df = pd.DataFrame({
    'X_Minimum': np.random.uniform(10, 100, n_samples),
    'X_Maximum': np.random.uniform(50, 150, n_samples),
    'Y_Minimum': np.random.uniform(10, 100, n_samples),
    'Y_Maximum': np.random.uniform(50, 150, n_samples),
    'Pixel_area': np.random.uniform(100, 500, n_samples),
    'Bare_Nuclei': np.random.uniform(0, 50, n_samples),
    'class': np.random.choice([1, 2, 3, 4, 5, 6, 7], n_samples)
})

print(f"Dataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst few rows:\n{df.head()}")
print(f"\nBasic statistics:\n{df.describe()}")

## Step 1: Load Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import mlflow
import mlflow.sklearn
import mlflow.keras
import joblib

print("Libraries imported successfully")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Environment configured")

# Quality Inspection - Steel Plates Faults EDA & Model Training

End-to-end exploratory data analysis and model training for the Steel Plates Faults dataset.

**Dataset**: Steel Plates Faults - Multi-class defect classification with 7 fault types  
**Models**: Decision Tree, SVM, XGBoost, ANN  
**Goal**: Build, train, evaluate, and persist models with MLflow integration